<a href="https://colab.research.google.com/github/thotasriharsha/ReinforcementLearning/blob/main/RL_ASS(7)_2159.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Assignment 1: Undiscounted and Discounted Return

n = int(input("Enter the number of time steps: "))

rewards = []
for t in range(1, n + 1):
    r = float(input(f"Enter reward for time step {t}: "))
    rewards.append(r)

gamma = float(input("Enter the discount factor gamma (0 to 1): "))

# Undiscounted return: G = R1 + R2 + ... + Rn
undiscounted_return = sum(rewards)

# Discounted return: G = R1 + gamma*R2 + gamma^2*R3 + ... + gamma^(n-1)*Rn
discounted_return = 0.0
for k, r in enumerate(rewards):
    term = (gamma ** k) * r
    print(f"  gamma^{k} * R{k + 1} = {gamma ** k:.4f} * {r} = {term:.4f}")
    discounted_return += term

print("\n----- Results -----")
print("Entered rewards     :", rewards)
print("Discount factor     :", gamma)
print("Undiscounted return :", round(undiscounted_return, 4))
print("Discounted return   :", round(discounted_return, 4))

Enter the number of time steps: 3
Enter reward for time step 1: 2
Enter reward for time step 2: 3
Enter reward for time step 3: 4
Enter the discount factor gamma (0 to 1): 2
  gamma^0 * R1 = 1.0000 * 2.0 = 2.0000
  gamma^1 * R2 = 2.0000 * 3.0 = 6.0000
  gamma^2 * R3 = 4.0000 * 4.0 = 16.0000

----- Results -----
Entered rewards     : [2.0, 3.0, 4.0]
Discount factor     : 2.0
Undiscounted return : 9.0
Discounted return   : 24.0


In [2]:
"""
grid_env_det.py
Deterministic 2 x 3 Grid World represented as an MDP.

    (0,0)=S0   (0,1)=S1   (0,2)=G
    (1,0)=X    (1,1)=S3   (1,2)=S4

X is an obstacle (not a valid state), G is the goal (terminal).
"""


class GridWorldDet:
    def __init__(self, gamma=0.9):
        # ---- State space (X is an obstacle, so it is NOT a state) ----
        self.states = ['S0', 'S1', 'S3', 'S4', 'G']
        self.terminal_states = ['G']
        self.state_pos = {'S0': (0, 0), 'S1': (0, 1), 'G': (0, 2),
                          'S3': (1, 1), 'S4': (1, 2)}
        self.pos_state = {pos: s for s, pos in self.state_pos.items()}
        self.obstacle = (1, 0)
        self.rows, self.cols = 2, 3

        # ---- Action space ----
        self.actions = ['up', 'down', 'left', 'right']
        self.action_delta = {'up': (-1, 0), 'down': (1, 0),
                             'left': (0, -1), 'right': (0, 1)}

        # ---- Rewards and discount factor ----
        self.step_reward = -1        # normal movement
        self.goal_reward = 10        # reaching G  (episode ends)
        self.obstacle_reward = -5    # stepping into X (episode ends)
        self.gamma = gamma

    # ---- helper: where would the agent land? (None = blocked by obstacle) ----
    def _target(self, state, action):
        r, c = self.state_pos[state]
        dr, dc = self.action_delta[action]
        nr, nc = r + dr, c + dc
        if (nr, nc) == self.obstacle:
            return 'X'
        if not (0 <= nr < self.rows and 0 <= nc < self.cols):
            return state                      # off the grid -> stay in place
        return self.pos_state[(nr, nc)]

    # ---- Transition function: s' = T(s, a)  (deterministic) ----
    def transition(self, state, action):
        if state in self.terminal_states:
            return state                      # G is absorbing
        target = self._target(state, action)
        return state if target == 'X' else target   # cannot move into obstacle

    # ---- Reward function: R(s, a) ----
    def reward(self, state, action):
        if state in self.terminal_states:
            return 0
        target = self._target(state, action)
        if target == 'X':
            return self.obstacle_reward
        if target == 'G':
            return self.goal_reward
        return self.step_reward

    # ---- Does the episode end after taking (s, a)? ----
    def is_done(self, state, action):
        if state in self.terminal_states:
            return True
        return self._target(state, action) in ('X', 'G')

    # ---- One environment step ----
    def step(self, state, action):
        return (self.transition(state, action),
                self.reward(state, action),
                self.is_done(state, action))

In [3]:
# Assignment 2: use the Grid World MDP defined in grid_env_det.py

env = GridWorldDet()

# a. State space
print("a. State space  :", env.states, " (X is an obstacle, not a state)")

# b. Action space
print("b. Action space :", env.actions)

# c. Reward function R(s, a)
print("\nc. Reward function R(s, a)")
print(f"{'State':<7}" + "".join(f"{a:<8}" for a in env.actions))
for s in env.states:
    if s in env.terminal_states:
        continue
    print(f"{s:<7}" + "".join(f"{env.reward(s, a):<8}" for a in env.actions))

# d. Discount factor
print("\nd. Discount factor gamma =", env.gamma)

# e. Transition function T(s, a) -> s'
print("\ne. Transition function T(s, a) -> s'")
print(f"{'State':<7}" + "".join(f"{a:<8}" for a in env.actions))
for s in env.states:
    if s in env.terminal_states:
        continue
    print(f"{s:<7}" + "".join(f"{env.transition(s, a):<8}" for a in env.actions))

# f. Estimate next state for a user-given (state, action)
print("\nf. Next-state estimation")
state = input("Enter current state (S0, S1, S3, S4): ").strip().upper()
action = input("Enter action (up, down, left, right): ").strip().lower()

if state not in env.states or state in env.terminal_states or action not in env.actions:
    print("Invalid state or action.")
else:
    next_state, reward, done = env.step(state, action)
    print(f"T({state}, {action}) = {next_state},  reward = {reward},  episode done = {done}")

# Example episode: S0 -> right -> S1 -> right -> G, with its returns
print("\nExample episode: S0 -> right -> S1 -> right -> G")
s, rewards, done = 'S0', [], False
for a in ['right', 'right']:
    s_next, r, done = env.step(s, a)
    print(f"  ({s}, {a}) -> {s_next}, reward = {r}")
    rewards.append(r)
    s = s_next
G_undisc = sum(rewards)
G_disc = sum((env.gamma ** k) * r for k, r in enumerate(rewards))
print("  Rewards:", rewards)
print("  Undiscounted return:", G_undisc)
print("  Discounted return  :", round(G_disc, 4))

a. State space  : ['S0', 'S1', 'S3', 'S4', 'G']  (X is an obstacle, not a state)
b. Action space : ['up', 'down', 'left', 'right']

c. Reward function R(s, a)
State  up      down    left    right   
S0     -1      -5      -1      -1      
S1     -1      -1      -1      10      
S3     -1      -1      -5      -1      
S4     10      -1      -1      -1      

d. Discount factor gamma = 0.9

e. Transition function T(s, a) -> s'
State  up      down    left    right   
S0     S0      S0      S0      S1      
S1     S1      S3      S0      G       
S3     S1      S3      S3      S4      
S4     G       S4      S3      S4      

f. Next-state estimation
Enter current state (S0, S1, S3, S4): 1, 2, 3, 4
Enter action (up, down, left, right): up, down, left, right
Invalid state or action.

Example episode: S0 -> right -> S1 -> right -> G
  (S0, right) -> S1, reward = -1
  (S1, right) -> G, reward = 10
  Rewards: [-1, 10]
  Undiscounted return: 9
  Discounted return  : 8.0
